# Kentucky County Data Explorer

Explore KyFromAbove data availability across Kentucky's 120 counties.
This notebook demonstrates how to:
- List all available counties
- Search multiple products for a single county
- Compare tile coverage across products
- Export results to GeoJSON

In [ ]:
import matplotlib.pyplot as plt

import abovepy
from abovepy.utils.bbox import get_county_bbox, list_counties

## List All Counties

abovepy ships with bounding boxes for all 120 Kentucky counties.
These are used internally when you pass `county=` to `abovepy.search()`.

In [ ]:
counties = list_counties()
print(f"Total counties: {len(counties)}\n")

# Display in columns for readability
cols = 4
for i in range(0, len(counties), cols):
    row = counties[i:i + cols]
    print("  ".join(f"{c:<16}" for c in row))

## Search Multiple Products for One County

Let's see what data is available for Pike County — the largest county by area
in eastern Kentucky. We'll search across DEM, orthoimagery, and LiDAR products.

In [ ]:
county_name = "Pike"
print(f"Searching {county_name} County...")
print(f"Bounding box: {get_county_bbox(county_name)}\n")

products_to_search = [
    "dem_phase1",
    "dem_phase2",
    "dem_phase3",
    "ortho_phase1",
    "ortho_phase2",
    "ortho_phase3",
    "laz_phase1",
    "laz_phase2",
    "laz_phase3",
]

results = {}
for product in products_to_search:
    try:
        tiles = abovepy.search(county=county_name, product=product)
        results[product] = tiles
        print(f"  {product:<16} {len(tiles):>5} tiles")
    except Exception as e:
        print(f"  {product:<16} Error: {e}")

print(f"\nTotal products with data: {len(results)}")

## Compare Tile Coverage

Visualize the spatial coverage of different products over Pike County
by plotting the tile boundaries from each search result.

In [ ]:
# Pick up to 3 products that returned results for comparison
products_to_plot = [k for k in ["dem_phase3", "ortho_phase3", "laz_phase3"] if k in results]

if products_to_plot:
    fig, axes = plt.subplots(
        1, len(products_to_plot),
        figsize=(6 * len(products_to_plot), 6),
        squeeze=False,
    )

    for idx, product in enumerate(products_to_plot):
        ax = axes[0][idx]
        gdf = results[product]
        gdf.plot(ax=ax, edgecolor="steelblue", facecolor="lightblue", alpha=0.5, linewidth=0.5)
        ax.set_title(f"{product}\n({len(gdf)} tiles)")
        ax.set_xlabel("Easting")
        ax.set_ylabel("Northing")

    plt.suptitle(f"{county_name} County — Tile Coverage by Product", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No products returned results for plotting.")

## Export Results

Export the tile index for a product to GeoJSON for use in other tools
(QGIS, ArcGIS Pro, web maps, etc.).

In [ ]:
from pathlib import Path

output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)

# Export each product's tile index to GeoJSON
for product, gdf in results.items():
    output_path = output_dir / f"{county_name.lower()}_{product}.geojson"
    gdf.to_file(output_path, driver="GeoJSON")
    print(f"Exported {product}: {output_path} ({len(gdf)} tiles)")

print(f"\nAll exports saved to {output_dir.resolve()}")